# External Test: Original Manually Captured Dataset

This notebook evaluates the already-trained Roboflow MobileNetV3Small model on the original manually captured dataset. The old dataset is external and test-only: it is not added to training, validation, or the Roboflow 8-fold splits.

The default mode is `raw_center_crop_224` (no segmentation), matching the current developer comparison. Set `TEST_INPUT_MODE` to `processed_hsv_lab_threshold_roi_224` to run the segmented comparison.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

from meatlens_pork_pipeline.image_ops import process_image
from meatlens_pork_pipeline.notebook_progress import iter_notebook_progress

NOTEBOOK_OVERRIDES = {
    'DATASET_SOURCE': 'current',
    'INPUT_MODE': 'raw_center_crop_224',
}

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

import onnxruntime as ort

LABEL_ORDER = ['fresh', 'not fresh', 'spoiled']
TEST_INPUT_MODE = str(override('TEST_INPUT_MODE', 'raw_center_crop_224')).strip()
if TEST_INPUT_MODE not in {'raw_center_crop_224', 'processed_hsv_lab_threshold_roi_224'}:
    raise ValueError('TEST_INPUT_MODE must be raw_center_crop_224 or processed_hsv_lab_threshold_roi_224.')

OLD_DATASET_MANIFEST_PATH = Path(str(override(
    'OLD_DATASET_MANIFEST_PATH',
    ROOT / 'generated_splits' / 'processed_manifest.csv',
)))
OLD_DATASET_ROOT = Path(str(override('OLD_DATASET_ROOT', ROOT / 'old_dataset')))
MODEL_PATH = Path(str(override(
    'MODEL_PATH',
    ROOT / 'training_outputs' / 'roboflow' /
    'mobilenetv3small_8samples_final_deployment_cnn_only' / 'models' /
    'meatlens_final_8samples_cnn_only_mobilenetv3small.onnx',
)))
OUTPUT_ROOT = Path(str(override(
    'EXTERNAL_TEST_OUTPUT_ROOT',
    ROOT / 'training_outputs' / 'external_old_dataset_test' /
    f'mobilenetv3small_roboflow_{TEST_INPUT_MODE}',
)))

if not OLD_DATASET_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Old-dataset manifest not found: {OLD_DATASET_MANIFEST_PATH}')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Roboflow ONNX model not found: {MODEL_PATH}')

print('External test mode:', TEST_INPUT_MODE)
print('Old-dataset manifest:', OLD_DATASET_MANIFEST_PATH)
print('Model:', MODEL_PATH)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
def _existing_path(value: object) -> Path | None:
    text = str(value or '').strip()
    if not text:
        return None
    candidate = Path(text)
    return candidate if candidate.exists() and candidate.is_file() else None


def resolve_old_dataset_image(row: dict[str, object], require_raw: bool) -> Path:
    file_name = str(row.get('image_file_name', row.get('image_name', ''))).strip()
    direct_candidates = [
        _existing_path(row.get('file_destination')),
        _existing_path(row.get('raw_image_path')),
    ]
    for candidate in direct_candidates:
        if candidate is not None:
            return candidate

    if OLD_DATASET_ROOT.exists() and file_name:
        matches = sorted(OLD_DATASET_ROOT.rglob(file_name))
        if matches:
            return matches[0]

    processed_candidate = _existing_path(row.get('local_image_path'))
    if not require_raw and processed_candidate is not None:
        return processed_candidate

    mode_hint = 'raw source images' if require_raw else 'source or processed images'
    raise FileNotFoundError(
        f'Could not resolve {mode_hint} for {file_name!r}. '
        f'Set OLD_DATASET_ROOT to the original manually captured dataset folder.'
    )


def load_external_image(path: Path, input_mode: str) -> tuple[np.ndarray, dict[str, object]]:
    if input_mode == 'processed_hsv_lab_threshold_roi_224':
        image_uint8, metadata = process_image(path, background_mode='gray')
        return image_uint8, metadata

    image = Image.open(path).convert('RGB')
    width, height = image.size
    side = min(width, height)
    left = (width - side) // 2
    top = (height - side) // 2
    image = image.crop((left, top, left + side, top + side)).resize(TARGET_SIZE, Image.BILINEAR)
    return np.asarray(image, dtype=np.uint8), {
        'segmentation_failed': False,
        'mask_area_ratio': '',
        'center_overlap_ratio': '',
        'number_of_components': '',
        'touches_border': '',
    }


manifest_df = pd.read_csv(OLD_DATASET_MANIFEST_PATH, dtype=str).fillna('')
manifest_df['label'] = manifest_df['label'].astype(str).str.strip().str.lower()
manifest_df = manifest_df[manifest_df['label'].isin(LABEL_ORDER)].reset_index(drop=True)
if manifest_df.empty:
    raise ValueError('The external old-dataset manifest contains no recognized labels.')

require_raw = TEST_INPUT_MODE == 'raw_center_crop_224'
resolved_paths = []
for row in manifest_df.to_dict(orient='records'):
    resolved_paths.append(resolve_old_dataset_image(row, require_raw=require_raw))
manifest_df['external_test_image_path'] = [str(path) for path in resolved_paths]
manifest_df['dataset_role'] = 'external_test_only'
manifest_df['test_input_mode'] = TEST_INPUT_MODE
print('External test images:', len(manifest_df))
print(manifest_df['label'].value_counts().reindex(LABEL_ORDER, fill_value=0).to_string())

In [ ]:
def prepare_model_batch(images_uint8: list[np.ndarray], channels_first: bool) -> np.ndarray:
    batch = np.stack(images_uint8, axis=0).astype(np.float32)
    batch = (batch / 127.5) - 1.0
    return np.transpose(batch, (0, 3, 1, 2)) if channels_first else batch


def summarize_external_predictions(prediction_df: pd.DataFrame) -> tuple[dict[str, object], np.ndarray]:
    y_true = prediction_df['true_label'].astype(str)
    y_pred = prediction_df['predicted_label'].astype(str)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average='macro', zero_division=0
    )
    confusion = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    metrics = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'test_count': int(len(prediction_df)),
        'label_order': LABEL_ORDER,
        'dataset_role': 'external_test_only',
        'training_dataset': 'roboflow',
        'test_dataset': 'current_manually_captured',
        'test_input_mode': TEST_INPUT_MODE,
        'model_path': str(MODEL_PATH),
    }
    return metrics, confusion


session = ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
model_input = session.get_inputs()[0]
model_shape = list(model_input.shape)
channels_first = len(model_shape) == 4 and model_shape[1] == 3

prediction_rows = []
for row in iter_notebook_progress(
    manifest_df.to_dict(orient='records'),
    '09_external_test_old_dataset.ipynb | external old-dataset inference',
    total=len(manifest_df),
    unit='image',
    leave=True,
):
    image_path = Path(str(row['external_test_image_path']))
    image_uint8, preprocessing_metadata = load_external_image(image_path, TEST_INPUT_MODE)
    batch = prepare_model_batch([image_uint8], channels_first=channels_first)
    probabilities = np.asarray(session.run(None, {model_input.name: batch})[0])[0]
    predicted_index = int(np.argmax(probabilities))
    prediction_rows.append({
        **row,
        **preprocessing_metadata,
        'true_label': str(row['label']),
        'predicted_label': LABEL_ORDER[predicted_index],
        'confidence': float(probabilities[predicted_index]),
        'fresh_probability': float(probabilities[0]),
        'not_fresh_probability': float(probabilities[1]),
        'spoiled_probability': float(probabilities[2]),
    })


prediction_df = pd.DataFrame(prediction_rows)
metrics, confusion = summarize_external_predictions(prediction_df)
print(json.dumps(metrics, indent=2))

In [ ]:
ensure_dir(OUTPUT_ROOT)
prediction_path = OUTPUT_ROOT / 'external_test_predictions.csv'
metrics_path = OUTPUT_ROOT / 'external_test_metrics.json'
confusion_path = OUTPUT_ROOT / 'external_test_confusion_matrix.csv'
summary_path = OUTPUT_ROOT / 'external_test_summary.json'

prediction_df.to_csv(prediction_path, index=False)
pd.DataFrame(confusion, index=LABEL_ORDER, columns=LABEL_ORDER).to_csv(confusion_path)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
summary_path.write_text(json.dumps({
    **metrics,
    'prediction_path': str(prediction_path),
    'confusion_matrix_path': str(confusion_path),
    'metrics_path': str(metrics_path),
}, indent=2), encoding='utf-8')

print('Saved external-test predictions:', prediction_path)
print('Saved external-test metrics:', metrics_path)
print('Saved confusion matrix:', confusion_path)
print(pd.DataFrame(confusion, index=LABEL_ORDER, columns=LABEL_ORDER))